In [3]:
# ==============================================
# 📊 Nifty 50 Stock Predictor - Backend Version
# Author: Akshay | Uses LSTM for stock prediction
# ==============================================


# !pip install yfinance tensorflow matplotlib scikit-learn pandas numpy tabulate --quiet

import warnings, os, logging
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("absl").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Input
from tabulate import tabulate

# ==============================================
# STEP 1️⃣ - Download Nifty 50 Data (Base Index)
# ==============================================
print("📥 Downloading Nifty 50 Index Data...")
nifty_symbol = "^NSEI"
data = yf.download(nifty_symbol, period="1y", interval="1d", progress=False)[['Close']].dropna()

# ==============================================
# STEP 2️⃣ - Preprocess Data
# ==============================================
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

def create_sequences(data, seq_len=60):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    return X, y

seq_len = 60
X, y = create_sequences(scaled_data, seq_len)

# ==============================================
# STEP 3️⃣ - Build / Load Model
# ==============================================
model_file = "lstm_model.h5"
if os.path.exists(model_file):
    print("✅ Loading existing model...")
    model = load_model(model_file)
else:
    print("🧠 Training new LSTM model...")
    model = Sequential([
        Input(shape=(seq_len, 1)),
        LSTM(50, return_sequences=True),
        LSTM(50),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X, y, epochs=10, batch_size=32, verbose=1)
    print("✅ Model training completed successfully.")
    model.save(model_file)
    print(f"💾 Model saved as {model_file}")

# ==============================================
# STEP 4️⃣ - Predict on All Nifty 50 Stocks
# ==============================================
print("\n🔮 Predicting Nifty 50 Stocks...\n")

nifty50_tickers = [
    "ADANIENT.NS", "ADANIPORTS.NS", "ASIANPAINT.NS", "AXISBANK.NS", "BAJAJ-AUTO.NS",
    "BAJFINANCE.NS", "BAJAJFINSV.NS", "BPCL.NS", "BRITANNIA.NS", "CIPLA.NS",
    "COALINDIA.NS", "DIVISLAB.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS",
    "HCLTECH.NS", "HDFC.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS",
    "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS", "INFY.NS",
    "IOC.NS", "ITC.NS", "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS",
    "M&M.NS", "MARUTI.NS", "NESTLEIND.NS", "NTPC.NS", "ONGC.NS",
    "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SBIN.NS", "SHREECEM.NS",
    "SUNPHARMA.NS", "TATACONSUM.NS", "TATAMOTORS.NS", "TATASTEEL.NS", "TCS.NS",
    "TECHM.NS", "ULTRACEMCO.NS", "UPL.NS", "WIPRO.NS"
]

results = []

for ticker in nifty50_tickers:
    try:
        stock = yf.download(ticker, period="6mo", interval="1d", progress=False)
        if len(stock) < seq_len:
            continue

        scaled = scaler.fit_transform(stock[['Close']])
        X_s = np.array([scaled[-seq_len:]])
        pred_s = model.predict(X_s, verbose=0)
        pred_price = scaler.inverse_transform(pred_s)[0][0]
        curr_price = stock['Close'].iloc[-1].item()

        if pred_price > curr_price * 1.01:
            signal = "BUY ✅"
        elif pred_price < curr_price * 0.99:
            signal = "SELL ❌"
        else:
            signal = "HOLD ⏸️"

        trend = "📈 Uptrend" if pred_price > curr_price else "📉 Downtrend"
        move_pct = ((pred_price - curr_price) / curr_price) * 100

        results.append([
            ticker.replace(".NS", ""),
            round(curr_price, 2),
            round(pred_price, 2),
            signal,
            trend,
            f"{move_pct:+.2f}%"
        ])
    except Exception as e:
        print(f"⚠️ Error with {ticker}: {e}")
        continue

# ==============================================
# STEP 5️⃣ - Display Results (Improved Table)
# ==============================================
df_results = pd.DataFrame(results, columns=[
    "Stock", "Current Price", "Predicted Price", "Signal", "Trend", "Predicted Move (%)"
])

print("\n✅ Predictions Complete!\n")
print(tabulate(df_results, headers='keys', tablefmt='fancy_grid', showindex=False))

# Save to CSV
df_results.to_csv("nifty50_predictions.csv", index=False)
print("\n📁 Results saved to 'nifty50_predictions.csv'")


📥 Downloading Nifty 50 Index Data...
✅ Loading existing model...

🔮 Predicting Nifty 50 Stocks...


✅ Predictions Complete!

╒════════════╤═════════════════╤═══════════════════╤══════════╤══════════════╤══════════════════════╕
│ Stock      │   Current Price │   Predicted Price │ Signal   │ Trend        │ Predicted Move (%)   │
╞════════════╪═════════════════╪═══════════════════╪══════════╪══════════════╪══════════════════════╡
│ ADANIENT   │         2549.4  │           2542    │ HOLD ⏸️  │ 📉 Downtrend │ -0.29%               │
├────────────┼─────────────────┼───────────────────┼──────────┼──────────────┼──────────────────────┤
│ ADANIPORTS │         1479.4  │           1419.12 │ SELL ❌  │ 📉 Downtrend │ -4.07%               │
├────────────┼─────────────────┼───────────────────┼──────────┼──────────────┼──────────────────────┤
│ ASIANPAINT │         2507.8  │           2380.35 │ SELL ❌  │ 📉 Downtrend │ -5.08%               │
├────────────┼─────────────────┼───────────────────┼──────────┼─